# Inferência no CRAI05

No VS Code Remote-SSH, abre a pasta do clone existente e seleciona um kernel Python **remoto** com as dependências do projeto. No terminal remoto, usa `nvidia-smi -L` para identificar as GPUs e `nvidia-smi` para consultar a ocupação.

Executa a célula de configuração e depois a célula de qualquer modelo. Corre um modelo de cada vez para evitar concorrência pela GPU e pela escrita no CSV. Cada modelo corre num processo separado com o Python do kernel.

In [ ]:
import os
import shlex
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_PATH = None  # Se necessário, indica o caminho absoluto do clone remoto.
GPU_INDEX = 0  # Índice dentro de CUDA_VISIBLE_DEVICES, se definido.
HF_CACHE = None  # Usa a cache do ambiente ou indica o caminho de HF_HUB_CACHE.
LOCAL_CHECKPOINTS_ONLY = True
BATCH_SIZE = 1
WARMUP_BATCHES = 1
MEASURED_BATCHES = 5

cwd = Path.cwd().resolve()
candidates = (cwd, *cwd.parents) if REPO_PATH is None else (
    Path(REPO_PATH).expanduser().resolve(),
)
repo = next(
    (path for path in candidates if (path / "rae_experiment/sampling/performance.py").is_file()),
    None,
)
if repo is None:
    raise RuntimeError("Define REPO_PATH com o caminho do clone existente no CRAI05.")

env = os.environ.copy()
gpu_mask = env.get("CUDA_VISIBLE_DEVICES")
devices = gpu_mask.split(",") if gpu_mask is not None else None
if GPU_INDEX < 0 or (devices is not None and GPU_INDEX >= len(devices)):
    raise ValueError("GPU_INDEX está fora dos índices disponíveis.")
if devices is None:
    env.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
env["CUDA_VISIBLE_DEVICES"] = str(GPU_INDEX) if devices is None else devices[GPU_INDEX].strip()

if HF_CACHE is not None:
    env["HF_HUB_CACHE"] = str(Path(HF_CACHE).expanduser().resolve())
env["HF_HUB_OFFLINE"] = "1" if LOCAL_CHECKPOINTS_ONLY else "0"

gpu_probe = """
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise SystemExit("A seleção deve expor uma GPU CUDA ao processo.")
print(torch.cuda.get_device_name(0))
print(int(torch.cuda.is_bf16_supported(including_emulation=False)))
"""
probe = subprocess.run(
    [sys.executable, "-c", gpu_probe], cwd=repo, env=env,
    check=True, stdout=subprocess.PIPE, text=True,
)
gpu_name, bf16 = probe.stdout.strip().rsplit("\n", 1)
gpu_name = gpu_name.strip()
if "RTX A6000" not in gpu_name:
    raise RuntimeError(f"GPU selecionada: {gpu_name}. Ajusta GPU_INDEX para selecionar uma RTX A6000.")

sana_precision = "bf16" if bf16 == "1" else "fp16"
results = repo / "outputs" / "crai05"
csv_path = results / "generation_performance.csv"
print(f"Python: {sys.executable}\nClone: {repo}\nGPU: {gpu_name} (cuda:0)")
print(f"Precisão SANA e SANA-Sprint: {sana_precision}\nCSV: {csv_path}")


def run_model(module, slug, precision, dependency_imports):
    preflight = "import torch, torchvision, PIL, huggingface_hub\n" + dependency_imports
    subprocess.run([sys.executable, "-c", preflight], cwd=repo, env=env, check=True)

    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output_dir = results / slug / run_id
    command = [
        sys.executable, "-m", f"rae_experiment.sampling.{module}",
        "--precision", precision,
        "--batch-size", str(BATCH_SIZE),
        "--warmup-batches", str(WARMUP_BATCHES),
        "--batches", str(MEASURED_BATCHES),
        "--csv", str(csv_path),
        "--output-dir", str(output_dir),
    ]
    env_args = [
        f"{key}={env[key]}" for key in
        ("CUDA_DEVICE_ORDER", "CUDA_VISIBLE_DEVICES", "HF_HUB_OFFLINE", "HF_HUB_CACHE")
        if key in env
    ]
    print(f"cd {shlex.quote(str(repo))} && {shlex.join(['env', *env_args, *command])}", flush=True)
    subprocess.run(command, cwd=repo, env=env, check=True)

A verificação inicial identifica a GPU e o suporte nativo de BF16. A célula de cada modelo verifica os seus imports antes de chamar a CLI. Em caso de erro de importação, confirma o kernel e compara as dependências com `requirements.txt` e, para o BAR, com `vendor/BAR/requirements.txt`.

Os scripts resolvem os seus checkpoints oficiais na cache Hugging Face. Com `LOCAL_CHECKPOINTS_ONLY = True`, a cache tem de conter todos os ficheiros necessários, incluindo o SigLIP2 `google/siglip2-so400m-patch16-naflex` para o BAR e os ficheiros do pipeline, scheduler e transformer da revisão de pMF fixada no script. Usa `HF_CACHE` para apontar para uma cache existente. Se quiseres permitir que as CLIs obtenham ficheiros em falta numa execução futura, altera `LOCAL_CHECKPOINTS_ONLY` para `False` e volta a executar a configuração.

As CLIs registam o tempo de carregamento, a latência, o throughput e o pico de VRAM no mesmo CSV. Os PNG medidos ficam numa pasta por modelo e execução. O notebook deixa os parâmetros de sampling e guidance ao cuidado das CLIs e mantém a KV-cache do BAR ativa.

## SANA-0.6B-512

In [ ]:
run_model(
    "generate_sana", "sana-0.6b-512", sana_precision,
    "import accelerate, sentencepiece\nfrom diffusers import SanaPipeline",
)

## SANA-Sprint-0.6B-1024

In [ ]:
run_model(
    "generate_sana_sprint", "sana-sprint-0.6b-1024", sana_precision,
    "import accelerate, sentencepiece\nfrom diffusers import SanaSprintPipeline",
)

## BAR-L-res512

FP16 no gerador através do autocast existente. O script conserva o decoder em FP32 e converte os logits de guidance e sampling para FP32.

In [ ]:
bar_root = repo / "vendor" / "BAR"
if not (bar_root / "configs/generator/bar_l_res512.yaml").is_file():
    raise FileNotFoundError("Falta vendor/BAR. No clone remoto, corre git submodule update --init vendor/BAR.")

run_model(
    "generate_bar", "bar-l-res512", "fp16",
    f"import sys\nsys.path.insert(0, {str(bar_root)!r})\n"
    "from modeling.generator import BAR\n"
    "from utils.train_utils import get_pretrained_tokenizer",
)

## pMF-L/32-512

FP32, a precisão por omissão da CLI. O script usa o scheduler publicado com a variante `pMF-L-32`.

In [ ]:
run_model(
    "generate_pmf", "pmf-l-32-512", "fp32",
    "import accelerate, safetensors\n"
    "from diffusers.utils.dynamic_modules_utils import get_class_from_dynamic_module",
)